In [2]:
#!/usr/bin/env python3
"""
╔══════════════════════════════════════════════════════╗
║   NETPULSE — Network Performance Monitor (Gradio)    ║
╚══════════════════════════════════════════════════════╝

Dependencies:
    pip install gradio psutil requests plotly

Run:
    python network_monitor_gradio.py
"""

import subprocess
import platform
import socket
import time
import re
import threading
from datetime import datetime

# ── Optional imports ──────────────────────────────────────────────────────────
try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

import gradio as gr

# ─────────────────────────────────────────────────────────────────────────────
# GLOBAL STATE  (bandwidth history for live charts)
# ─────────────────────────────────────────────────────────────────────────────
_bw_lock        = threading.Lock()
_dl_history:  list[float] = []
_ul_history:  list[float] = []
_time_labels: list[str]   = []
_MAX_POINTS = 60

_prev_io   = psutil.net_io_counters() if HAS_PSUTIL else None
_prev_time = time.time()


def _sample_bandwidth():
    """Take one bandwidth sample; returns (dl_bps, ul_bps)."""
    global _prev_io, _prev_time
    if not HAS_PSUTIL:
        return 0.0, 0.0
    now  = time.time()
    curr = psutil.net_io_counters()
    dt   = now - _prev_time or 1
    dl   = (curr.bytes_recv - _prev_io.bytes_recv) / dt
    ul   = (curr.bytes_sent - _prev_io.bytes_sent) / dt
    _prev_io   = curr
    _prev_time = now
    return max(dl, 0), max(ul, 0)


def _push_bw(dl, ul):
    with _bw_lock:
        ts = datetime.now().strftime("%H:%M:%S")
        _dl_history.append(dl)
        _ul_history.append(ul)
        _time_labels.append(ts)
        if len(_dl_history) > _MAX_POINTS:
            _dl_history.pop(0)
            _ul_history.pop(0)
            _time_labels.pop(0)


# ─────────────────────────────────────────────────────────────────────────────
# NETWORK UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
def fmt_bytes(n: float) -> str:
    for u in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} TB"


def run_ping(host: str, count: int = 4) -> tuple[bool, str]:
    sys = platform.system().lower()
    cmd = (["ping", "-n", str(count), host]
           if sys == "windows"
           else ["ping", "-c", str(count), "-W", "2", host])
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        return r.returncode == 0, r.stdout + r.stderr
    except subprocess.TimeoutExpired:
        return False, f"❌ Ping to {host} timed out."
    except FileNotFoundError:
        return False, "❌ ping command not found."
    except Exception as e:
        return False, f"❌ {e}"


def parse_latency(output: str) -> str | None:
    m = re.search(r"Average\s*=\s*(\d+)ms", output, re.I)
    if m: return m.group(1)
    m = re.search(r"min/avg/max.*?=\s*[\d.]+/([\d.]+)/", output)
    if m: return m.group(1)
    m = re.search(r"time[=<]([\d.]+)\s*ms", output, re.I)
    if m: return m.group(1)
    return None


def parse_loss(output: str) -> str | None:
    m = re.search(r"(\d+)%\s+packet loss", output, re.I)
    if m: return m.group(1) + "%"
    m = re.search(r"(\d+)%\s+loss", output, re.I)
    if m: return m.group(1) + "%"
    return None


def run_traceroute(host: str) -> tuple[bool, str]:
    sys = platform.system().lower()
    cmd = (["tracert", "-h", "15", host]
           if sys == "windows"
           else ["traceroute", "-m", "15", "-w", "2", host])
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        return True, r.stdout + r.stderr
    except subprocess.TimeoutExpired:
        return False, "❌ Traceroute timed out."
    except FileNotFoundError:
        tool = "tracert" if sys == "windows" else "traceroute"
        return False, f"❌ '{tool}' not found on this system."
    except Exception as e:
        return False, f"❌ {e}"


def dns_lookup(host: str) -> tuple[bool, list[str], str]:
    try:
        results = socket.getaddrinfo(host, None)
        ips = sorted({r[4][0] for r in results})
        return True, ips, ""
    except socket.gaierror as e:
        return False, [], str(e)


def get_interfaces() -> list[dict]:
    rows = []
    if HAS_PSUTIL:
        stats = psutil.net_if_stats()
        addrs = psutil.net_if_addrs()
        for name, addr_list in addrs.items():
            ipv4 = next((a.address for a in addr_list if a.family == socket.AF_INET), "—")
            ipv6 = next((a.address for a in addr_list if a.family == socket.AF_INET6), "—")
            st   = stats.get(name)
            up   = ("🟢 UP" if st and st.isup else "🔴 DOWN") if st else "—"
            spd  = f"{st.speed} Mbps" if st and st.speed else "—"
            rows.append({"Interface": name, "Status": up, "IPv4": ipv4,
                         "IPv6": ipv6[:30] + "…" if len(ipv6) > 30 else ipv6,
                         "Speed": spd})
    else:
        try:
            hostname = socket.gethostname()
            ip = socket.gethostbyname(hostname)
            rows.append({"Interface": "default", "Status": "🟢 UP",
                         "IPv4": ip, "IPv6": "—", "Speed": "—"})
        except Exception:
            rows.append({"Interface": "N/A", "Status": "—",
                         "IPv4": "—", "IPv6": "—", "Speed": "—"})
    return rows


def system_info() -> dict:
    info = {
        "Hostname":   socket.gethostname(),
        "OS":         platform.system() + " " + platform.release(),
        "Python":     platform.python_version(),
        "psutil":     "✅ installed" if HAS_PSUTIL else "❌ not installed (pip install psutil)",
        "plotly":     "✅ installed" if HAS_PLOTLY else "❌ not installed (pip install plotly)",
        "Local IP":   "—",
    }
    try:
        info["Local IP"] = socket.gethostbyname(socket.gethostname())
    except Exception:
        pass
    return info


# ─────────────────────────────────────────────────────────────────────────────
# CHART BUILDERS
# ─────────────────────────────────────────────────────────────────────────────
def _bw_chart():
    with _bw_lock:
        labels = list(_time_labels)
        dl     = list(_dl_history)
        ul     = list(_ul_history)

    if not HAS_PLOTLY:
        return None

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=labels, y=dl, name="↓ Download",
        line=dict(color="#00D4FF", width=2),
        fill="tozeroy", fillcolor="rgba(0,212,255,0.08)",
        mode="lines",
    ))
    fig.add_trace(go.Scatter(
        x=labels, y=ul, name="↑ Upload",
        line=dict(color="#00FF9C", width=2),
        fill="tozeroy", fillcolor="rgba(0,255,156,0.08)",
        mode="lines",
    ))
    fig.update_layout(
        paper_bgcolor="#0A0E1A",
        plot_bgcolor="#111827",
        font=dict(family="Courier New", color="#94A3B8", size=11),
        margin=dict(l=50, r=20, t=30, b=40),
        legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(color="#E2E8F0")),
        xaxis=dict(gridcolor="#1F2D45", showgrid=True, tickfont=dict(size=9)),
        yaxis=dict(gridcolor="#1F2D45", showgrid=True,
                   tickformat=".0f", title="Bytes/s"),
        title=dict(text="Real-time Bandwidth", font=dict(color="#00D4FF", size=13)),
        hovermode="x unified",
    )
    return fig


def _ping_chart(host: str, times_ms: list[float]):
    if not HAS_PLOTLY or not times_ms:
        return None
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=[f"#{i+1}" for i in range(len(times_ms))],
        y=times_ms,
        marker_color=["#00D4FF" if t < 100 else "#FFB800" if t < 300 else "#FF4444"
                      for t in times_ms],
        name="RTT (ms)",
    ))
    avg = sum(times_ms) / len(times_ms)
    fig.add_hline(y=avg, line_dash="dot", line_color="#94A3B8",
                  annotation_text=f"avg {avg:.1f} ms",
                  annotation_font_color="#94A3B8")
    fig.update_layout(
        paper_bgcolor="#0A0E1A",
        plot_bgcolor="#111827",
        font=dict(family="Courier New", color="#94A3B8", size=11),
        margin=dict(l=50, r=20, t=40, b=40),
        xaxis=dict(gridcolor="#1F2D45", title="Packet"),
        yaxis=dict(gridcolor="#1F2D45", title="RTT (ms)"),
        title=dict(text=f"Ping RTT — {host}", font=dict(color="#00D4FF", size=13)),
    )
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# GRADIO HANDLER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

# ── Ping ─────────────────────────────────────────────────────────────────────
def handle_ping(host: str, count: int):
    host = host.strip() or "google.com"
    ok, output = run_ping(host, count)

    lat  = parse_latency(output) or "—"
    loss = parse_loss(output)    or ("0%" if ok else "100%")
    status_md = (
        f"### {'✅ Success' if ok else '❌ Failed'}\n\n"
        f"| Metric | Value |\n|---|---|\n"
        f"| Host | `{host}` |\n"
        f"| Packets | `{count}` |\n"
        f"| Avg Latency | `{lat} ms` |\n"
        f"| Packet Loss | `{loss}` |"
    )

    # Extract individual RTT values for chart
    rtt_values = [float(m) for m in re.findall(r"time[=<]([\d.]+)\s*ms", output, re.I)]
    chart = _ping_chart(host, rtt_values)

    return status_md, output, chart


# ── Traceroute ────────────────────────────────────────────────────────────────
def handle_traceroute(host: str):
    host = host.strip() or "google.com"
    ok, output = run_traceroute(host)

    # Parse hops
    hop_rows = []
    for line in output.splitlines():
        m = re.match(r"\s*(\d+)\s+(.+)", line)
        if m:
            hop_no = m.group(1)
            rest   = m.group(2).strip()
            times  = re.findall(r"([\d.]+)\s*ms", rest)
            avg_t  = f"{sum(float(t) for t in times)/len(times):.1f} ms" if times else "* * *"
            hop_rows.append([hop_no, avg_t, rest[:60]])

    hdr = f"### {'✅ Traceroute Complete' if ok else '⚠️ Traceroute (partial)'}\n\n"
    hdr += f"**Target:** `{host}` | **Hops found:** `{len(hop_rows)}`\n"
    return hdr, hop_rows, output


# ── DNS ───────────────────────────────────────────────────────────────────────
def handle_dns(host: str):
    host = host.strip() or "example.com"
    ok, ips, err = dns_lookup(host)

    rows = [[ip, "IPv6" if ":" in ip else "IPv4"] for ip in ips]
    if ok:
        status = (
            f"### ✅ DNS Resolved\n\n"
            f"**Domain:** `{host}`  \n"
            f"**Records found:** `{len(ips)}`\n\n"
            + "\n".join(f"- `{ip}`" for ip in ips)
        )
    else:
        status = f"### ❌ DNS Failed\n\n**Error:** {err}"
    return status, rows


# ── Multi-ping (Quick Check) ──────────────────────────────────────────────────
def handle_multi_ping(hosts_text: str):
    hosts = [h.strip() for h in hosts_text.splitlines() if h.strip()]
    if not hosts:
        hosts = ["8.8.8.8", "1.1.1.1", "google.com", "github.com"]

    rows = []
    for host in hosts:
        ok, output = run_ping(host, 2)
        lat  = parse_latency(output) or "—"
        loss = parse_loss(output)    or ("0%" if ok else "100%")
        status = "✅ UP" if ok else "❌ DOWN"
        rows.append([host, status, f"{lat} ms", loss])

    return rows


# ── Interfaces ────────────────────────────────────────────────────────────────
def handle_interfaces():
    return get_interfaces()


# ── Dashboard refresh ─────────────────────────────────────────────────────────
def handle_dashboard_refresh():
    dl, ul = _sample_bandwidth()
    _push_bw(dl, ul)

    # Heartbeat ping
    ok, out = run_ping("8.8.8.8", 1)
    lat  = parse_latency(out) or ("OK" if ok else "—")
    loss = parse_loss(out) or ("0%" if ok else "100%")

    info = system_info()

    metrics_md = (
        f"## 📡 Network Dashboard\n\n"
        f"| Metric | Value |\n|---|---|\n"
        f"| ↓ Download | **{fmt_bytes(dl)}/s** |\n"
        f"| ↑ Upload | **{fmt_bytes(ul)}/s** |\n"
        f"| Latency (8.8.8.8) | **{lat} ms** |\n"
        f"| Packet Loss | **{loss}** |\n"
        f"| Hostname | `{info['Hostname']}` |\n"
        f"| Local IP | `{info['Local IP']}` |\n"
        f"| OS | `{info['OS']}` |\n"
    )

    chart = _bw_chart()
    return metrics_md, chart


# ─────────────────────────────────────────────────────────────────────────────
# CUSTOM CSS  — cyber-industrial dark theme
# ─────────────────────────────────────────────────────────────────────────────
CSS = """
/* ── Imports ── */
@import url('https://fonts.googleapis.com/css2?family=Share+Tech+Mono&family=Exo+2:wght@300;600;800&display=swap');

/* ── Root ── */
:root {
    --bg:       #0A0E1B;
    --panel:    #111827;
    --border:   #1F2D45;
    --accent:   #00D4FF;
    --accent2:  #00FF9C;
    --warn:     #FFB800;
    --danger:   #FF4444;
    --text:     #E2E8F0;
    --muted:    #4A5568;
    --radius:   6px;
}

/* ── Global ── */
body, .gradio-container {
    background: var(--bg) !important;
    font-family: 'Share Tech Mono', monospace !important;
    color: var(--text) !important;
}

/* Header */
.app-header {
    background: linear-gradient(135deg, #0D1528 0%, #0A0E1A 60%, #0D1A2A 100%);
    border-bottom: 1px solid var(--border);
    padding: 20px 32px 16px;
    position: relative;
    overflow: hidden;
}
.app-header::before {
    content: '';
    position: absolute;
    inset: 0;
    background: repeating-linear-gradient(
        90deg,
        transparent,
        transparent 40px,
        rgba(0,212,255,0.03) 40px,
        rgba(0,212,255,0.03) 41px
    );
}
.app-title {
    font-family: 'Exo 2', sans-serif !important;
    font-size: 2rem !important;
    font-weight: 800 !important;
    color: var(--accent) !important;
    letter-spacing: 4px !important;
    text-shadow: 0 0 20px rgba(0,212,255,0.4) !important;
    margin: 0 !important;
}
.app-sub {
    color: var(--muted) !important;
    font-size: 0.8rem !important;
    letter-spacing: 2px !important;
    margin-top: 4px !important;
}

/* Tabs */
.tab-nav button {
    background: var(--panel) !important;
    color: var(--muted) !important;
    border: 1px solid var(--border) !important;
    border-bottom: none !important;
    font-family: 'Share Tech Mono', monospace !important;
    font-size: 0.8rem !important;
    letter-spacing: 1px !important;
    padding: 8px 18px !important;
    transition: all 0.2s !important;
}
.tab-nav button.selected {
    background: var(--border) !important;
    color: var(--accent) !important;
    border-color: var(--accent) !important;
    text-shadow: 0 0 8px rgba(0,212,255,0.5) !important;
}
.tab-nav button:hover:not(.selected) {
    color: var(--text) !important;
    border-color: var(--border) !important;
}

/* Panels / blocks */
.block, .gr-block, .form {
    background: var(--panel) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
}

/* Labels */
label span, .label-wrap span {
    color: var(--accent) !important;
    font-family: 'Share Tech Mono', monospace !important;
    font-size: 0.78rem !important;
    letter-spacing: 1px !important;
    text-transform: uppercase !important;
}

/* Inputs */
input[type=text], input[type=number], textarea {
    background: #0A0E1A !important;
    border: 1px solid var(--border) !important;
    color: var(--text) !important;
    font-family: 'Share Tech Mono', monospace !important;
    border-radius: var(--radius) !important;
    transition: border-color 0.2s !important;
}
input[type=text]:focus, textarea:focus {
    border-color: var(--accent) !important;
    box-shadow: 0 0 0 2px rgba(0,212,255,0.15) !important;
    outline: none !important;
}

/* Buttons */
button.primary {
    background: linear-gradient(135deg, var(--accent), #0099CC) !important;
    color: #0A0E1A !important;
    font-family: 'Share Tech Mono', monospace !important;
    font-weight: bold !important;
    letter-spacing: 1px !important;
    border: none !important;
    border-radius: var(--radius) !important;
    transition: all 0.2s !important;
    text-transform: uppercase !important;
}
button.primary:hover {
    background: linear-gradient(135deg, var(--accent2), #00CC7A) !important;
    box-shadow: 0 0 16px rgba(0,255,156,0.35) !important;
    transform: translateY(-1px) !important;
}
button.secondary {
    background: var(--border) !important;
    color: var(--accent) !important;
    border: 1px solid var(--accent) !important;
    font-family: 'Share Tech Mono', monospace !important;
    letter-spacing: 1px !important;
    border-radius: var(--radius) !important;
}
button.secondary:hover {
    background: rgba(0,212,255,0.1) !important;
    box-shadow: 0 0 12px rgba(0,212,255,0.2) !important;
}

/* Markdown output */
.prose, .md-output, [class*="markdown"] {
    color: var(--text) !important;
    font-family: 'Share Tech Mono', monospace !important;
}
.prose h2, .prose h3, [class*="markdown"] h2, [class*="markdown"] h3 {
    color: var(--accent) !important;
    border-bottom: 1px solid var(--border) !important;
    padding-bottom: 4px !important;
}
.prose table, [class*="markdown"] table {
    border-collapse: collapse !important;
    width: 100% !important;
}
.prose th, [class*="markdown"] th {
    background: var(--border) !important;
    color: var(--accent) !important;
    padding: 6px 12px !important;
    text-align: left !important;
}
.prose td, [class*="markdown"] td {
    border-bottom: 1px solid var(--border) !important;
    padding: 5px 12px !important;
    color: var(--text) !important;
}

/* Dataframe/table */
.table-wrap table thead th {
    background: var(--border) !important;
    color: var(--accent) !important;
    font-family: 'Share Tech Mono', monospace !important;
}
.table-wrap table tbody tr {
    background: var(--panel) !important;
    color: var(--text) !important;
}
.table-wrap table tbody tr:nth-child(even) {
    background: #0D1528 !important;
}
.table-wrap table tbody td {
    font-family: 'Share Tech Mono', monospace !important;
    border-color: var(--border) !important;
}

/* Textbox / code output */
.scroll-hide textarea {
    background: #06090F !important;
    color: #7FDBCA !important;
    font-family: 'Share Tech Mono', monospace !important;
    font-size: 0.82rem !important;
    border: 1px solid var(--border) !important;
    line-height: 1.6 !important;
}

/* Sliders */
input[type=range] {
    accent-color: var(--accent) !important;
}

/* Plotly chart container */
.js-plotly-plot {
    border-radius: var(--radius) !important;
}

/* Scrollbar */
::-webkit-scrollbar { width: 6px; height: 6px; }
::-webkit-scrollbar-track { background: var(--bg); }
::-webkit-scrollbar-thumb { background: var(--border); border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: var(--accent); }

/* Status badge */
.status-ok  { color: #00FF9C !important; font-weight: bold; }
.status-err { color: #FF4444 !important; font-weight: bold; }

/* Quick-check grid */
.quick-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; }
"""

# ─────────────────────────────────────────────────────────────────────────────
# BUILD GRADIO APP
# ─────────────────────────────────────────────────────────────────────────────
def build_app() -> gr.Blocks:
    with gr.Blocks(css=CSS, title="NetPulse — Network Monitor") as app:

        # ── Header ────────────────────────────────────────────────────────────
        gr.HTML("""
        <div class="app-header">
            <div class="app-title">◈ NETPULSE</div>
            <div class="app-sub">NETWORK PERFORMANCE MONITOR &nbsp;/&nbsp; REAL-TIME DIAGNOSTICS</div>
        </div>
        """)

        # ── Tabs ──────────────────────────────────────────────────────────────
        with gr.Tabs():

            # ── Dashboard ────────────────────────────────────────────────────
            with gr.Tab("◈ Dashboard"):
                dash_metrics = gr.Markdown("*Click **Refresh** to load metrics.*")
                dash_chart   = gr.Plot(label="Bandwidth History")

                with gr.Row():
                    dash_refresh = gr.Button("↺  Refresh Metrics", variant="primary")
                    gr.HTML("<span style='color:#4A5568;font-size:0.78rem;padding:10px 0;display:block;'>"
                            "Updates bandwidth, latency & packet loss. Click repeatedly for live graph.</span>")

                dash_refresh.click(
                    fn=handle_dashboard_refresh,
                    outputs=[dash_metrics, dash_chart],
                )

            # ── Ping ─────────────────────────────────────────────────────────
            with gr.Tab("⟳ Ping"):
                with gr.Row():
                    with gr.Column(scale=3):
                        ping_host  = gr.Textbox(label="Target Host / IP",
                                                value="google.com", placeholder="e.g. 8.8.8.8")
                    with gr.Column(scale=1):
                        ping_count = gr.Slider(label="Packet Count",
                                               minimum=1, maximum=20, value=4, step=1)
                    with gr.Column(scale=1):
                        ping_btn = gr.Button("▶  Run Ping", variant="primary")

                ping_status = gr.Markdown()
                ping_chart  = gr.Plot(label="Round-Trip Time per Packet")
                ping_raw    = gr.Textbox(label="Raw Output", lines=14, interactive=False)

                ping_btn.click(
                    fn=handle_ping,
                    inputs=[ping_host, ping_count],
                    outputs=[ping_status, ping_raw, ping_chart],
                )

            # ── Traceroute ───────────────────────────────────────────────────
            with gr.Tab("⤴ Traceroute"):
                with gr.Row():
                    with gr.Column(scale=4):
                        trace_host = gr.Textbox(label="Target Host / IP",
                                                value="google.com", placeholder="e.g. cloudflare.com")
                    with gr.Column(scale=1):
                        trace_btn = gr.Button("▶  Run Traceroute", variant="primary")

                trace_status = gr.Markdown()
                trace_table  = gr.Dataframe(
                    headers=["Hop", "Avg RTT", "Details"],
                    label="Hop Table",
                    interactive=False,
                )
                trace_raw = gr.Textbox(label="Raw Output", lines=16, interactive=False)

                trace_btn.click(
                    fn=handle_traceroute,
                    inputs=[trace_host],
                    outputs=[trace_status, trace_table, trace_raw],
                )

            # ── DNS Lookup ───────────────────────────────────────────────────
            with gr.Tab("⧫ DNS Lookup"):
                with gr.Row():
                    with gr.Column(scale=4):
                        dns_host = gr.Textbox(label="Domain / Hostname",
                                              value="example.com", placeholder="e.g. github.com")
                    with gr.Column(scale=1):
                        dns_btn = gr.Button("▶  Lookup", variant="primary")

                dns_status = gr.Markdown()
                dns_table  = gr.Dataframe(
                    headers=["IP Address", "Type"],
                    label="DNS Records",
                    interactive=False,
                )

                dns_btn.click(
                    fn=handle_dns,
                    inputs=[dns_host],
                    outputs=[dns_status, dns_table],
                )

            # ── Multi-host Check ─────────────────────────────────────────────
            with gr.Tab("⬡ Multi-Host Check"):
                gr.Markdown("Enter one host per line. Leave blank to use defaults.")
                multi_hosts = gr.Textbox(
                    label="Hosts (one per line)",
                    value="8.8.8.8\n1.1.1.1\ngoogle.com\ngithub.com",
                    lines=6,
                    placeholder="8.8.8.8\ncloudflare.com\n...",
                )
                multi_btn = gr.Button("▶  Check All Hosts", variant="primary")
                multi_table = gr.Dataframe(
                    headers=["Host", "Status", "Latency", "Packet Loss"],
                    label="Results",
                    interactive=False,
                )

                multi_btn.click(
                    fn=handle_multi_ping,
                    inputs=[multi_hosts],
                    outputs=[multi_table],
                )

            # ── Interfaces ───────────────────────────────────────────────────
            with gr.Tab("⬡ Interfaces"):
                iface_btn = gr.Button("↺  Refresh Interfaces", variant="secondary")
                iface_table = gr.Dataframe(
                    headers=["Interface", "Status", "IPv4", "IPv6", "Speed"],
                    label="Network Interfaces",
                    interactive=False,
                )

                iface_btn.click(fn=handle_interfaces, outputs=[iface_table])

                # Auto-load on start
                app.load(fn=handle_interfaces, outputs=[iface_table])

        # ── Footer ────────────────────────────────────────────────────────────
        gr.HTML("""
        <div style="
            background: #0D1528;
            border-top: 1px solid #1F2D45;
            padding: 12px 24px;
            margin-top: 16px;
            display: flex;
            justify-content: space-between;
            align-items: center;
        ">
            <span style="color:#4A5568;font-family:'Share Tech Mono',monospace;font-size:0.75rem;">
                ◈ NETPULSE — Network Performance Monitor
            </span>
            <span style="color:#4A5568;font-family:'Share Tech Mono',monospace;font-size:0.75rem;">
                Built with Python + Gradio
            </span>
        </div>
        """)

    return app





# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("""
╔══════════════════════════════════════════════════════╗
║   Live — Network Performance Monitor (Gradio)    ║
╚══════════════════════════════════════════════════════╝

  Dependencies: pip install gradio psutil plotly

  Starting server...
""")
    app = build_app()

app.launch(
    server_name="0.0.0.0",
    server_port=7861,  # changed port
    show_error=True,
    inbrowser=True,
    css="style.css"   # moved here if used
)


╔══════════════════════════════════════════════════════╗
║   Live — Network Performance Monitor (Gradio)    ║
╚══════════════════════════════════════════════════════╝

  Dependencies: pip install gradio psutil plotly

  Starting server...



C:\Users\Dell\AppData\Local\Temp\ipykernel_13272\1934348372.py:603: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, title="NetPulse — Network Monitor") as app:
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 7861): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted


OSError: Cannot find empty port in range: 7861-7861. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.